# Group 3

In [3]:
import os
import clip
import torch
from torchvision.datasets import CIFAR100

# Load the model
device = "cuda" if torch.cuda.is_available() else "cpu"
model, preprocess = clip.load('ViT-B/32', device)

# Download the dataset
cifar100 = CIFAR100(root=os.path.expanduser("~/.cache"), download=True, train=False)

# Prepare the inputs
image, class_id = cifar100[3637]
image_input = preprocess(image).unsqueeze(0).to(device)
text_inputs = torch.cat([clip.tokenize(f"a photo of a {c}") for c in cifar100.classes]).to(device)

# Calculate features
with torch.no_grad():
    image_features = model.encode_image(image_input)
    text_features = model.encode_text(text_inputs)

# Pick the top 5 most similar labels for the image
image_features /= image_features.norm(dim=-1, keepdim=True)
text_features /= text_features.norm(dim=-1, keepdim=True)
similarity = (100.0 * image_features @ text_features.T).softmax(dim=-1)
values, indices = similarity[0].topk(5)

# Print the result
print("\nTop predictions:\n")
for value, index in zip(values, indices):
    print(f"{cifar100.classes[index]:>16s}: {100 * value.item():.2f}%")

100%|██████████| 169M/169M [00:03<00:00, 42.5MB/s]



Top predictions:

           snake: 65.72%
          turtle: 12.16%
    sweet_pepper: 3.89%
          lizard: 1.89%
       crocodile: 1.72%


In [5]:
import torch
import clip
import numpy as np

from torchvision.datasets import FakeData
from torch.utils.data import DataLoader
from sklearn.linear_model import LogisticRegression

# Setup Device and Model
device = "cuda" if torch.cuda.is_available() else "cpu"
model, preprocess = clip.load("ViT-B/32", device=device)

# Create the FakeData Dataset
# We'll create a small "few-shot" training set and a small test set
num_classes = 250
samples_per_class = 20  # This makes it a "5-shot" problem

train_dataset = FakeData(
    size=num_classes * samples_per_class,
    image_size=(3, 224, 224),
    num_classes=num_classes,
    transform=preprocess
)

test_dataset = FakeData(
    size=50,
    image_size=(3, 224, 224),
    num_classes=num_classes,
    transform=preprocess
)

# Helper function to extract features
def extract_features(dataset):
    all_features = []
    all_labels = []

    loader = DataLoader(dataset, batch_size=32, shuffle=False)

    with torch.no_grad():
        for images, labels in loader:
            features = model.encode_image(images.to(device))
            # Normalize features for better stability
            features /= features.norm(dim=-1, keepdim=True)

            all_features.append(features.cpu().numpy())
            all_labels.append(labels.numpy())

    return np.concatenate(all_features), np.concatenate(all_labels)

# Process Data
print("Extracting features from FakeData...")
X_train, y_train = extract_features(train_dataset)
X_test, y_test = extract_features(test_dataset)

# Train the "Small Fine-Tuning" (Linear Probe)
# Using LogisticRegression as our lightweight head
classifier = LogisticRegression(max_iter=1000, C=1.0)
classifier.fit(X_train, y_train)

# 6. Evaluate
accuracy = classifier.score(X_test, y_test)
print(f"Few-Shot Accuracy on FakeData: {accuracy * 100:.2f}%")

Extracting features from FakeData...
Few-Shot Accuracy on FakeData: 0.00%


In [6]:
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader, BatchSampler

class BalancedBatchSampler(BatchSampler):
    """
    BatchSampler - from a MNIST-like dataset, samples n_classes and within these classes samples n_samples.
    Returns batches of size n_classes * n_samples
    """

    def __init__(self, labels, n_classes, n_samples):
        # label: unique id per datapoint, e.g. path
        self.labels = labels
        self.labels_set = list(set(self.labels.numpy()))
        self.label_to_indices = {label: np.where(self.labels.numpy() == label)[0]
                                 for label in self.labels_set}
        for l in self.labels_set:
            np.random.shuffle(self.label_to_indices[l])
        self.used_label_indices_count = {label: 0 for label in self.labels_set}
        self.count = 0
        self.n_classes = n_classes
        self.n_samples = n_samples
        self.n_dataset = len(self.labels)
        self.batch_size = self.n_samples * self.n_classes

    def __iter__(self):
        self.count = 0
        while self.count + self.batch_size < self.n_dataset:
            classes = np.random.choice(self.labels_set, self.n_classes, replace=False)
            indices = []
            for class_ in classes:
                indices.extend(self.label_to_indices[class_][
                               self.used_label_indices_count[class_]:self.used_label_indices_count[
                                                                         class_] + self.n_samples])
                self.used_label_indices_count[class_] += self.n_samples
                if self.used_label_indices_count[class_] + self.n_samples > len(self.label_to_indices[class_]):
                    np.random.shuffle(self.label_to_indices[class_])
                    self.used_label_indices_count[class_] = 0
            yield indices
            self.count += self.n_classes * self.n_samples

    def __len__(self):
        return self.n_dataset // self.batch_size

In [11]:
import random
import clip
import numpy as np
import torch
import torchmetrics
import torchvision

from pathlib import Path
from tqdm import tqdm
from torchvision.datasets import CIFAR10
from torch.utils.tensorboard import SummaryWriter


def finetune():
    torch.manual_seed(0)
    random.seed(0)
    np.random.seed(0)

    SAVE_INTERVAL = 10
    BATCH_SIZE = 8
    NUM_EPOCHS = 100

    def convert_models_to_fp32(model):
        for p in model.parameters():
            p.data = p.data.float()
            if p.requires_grad:
                p.grad.data = p.grad.data.float()

    # Setup Device and Model
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model, preprocess = clip.load("ViT-B/32", device=device, jit=False)  #Must set jit=False for training
    if device == "cpu":
        model.float()
    else:
        clip.model.convert_weights(model)  # Actually this line is unnecessary since clip by default already on float16

    writer = SummaryWriter()
    weights_path = Path("model_checkpoints")
    weights_path.mkdir(exist_ok=True)

    # Download the dataset
    train_dataset = CIFAR10(root=os.path.expanduser("~/.cache"), download=True, train=True)
    train_labels = torch.tensor(train_dataset.targets)
    train_sampler = BalancedBatchSampler(train_labels, BATCH_SIZE, 1)
    # use drop_last = True to ensure each batch contains 8 target classes to choose from.
    train_dataloader = torch.utils.data.DataLoader(train_dataset, drop_last=True)

    test_dataset = CIFAR10(root=os.path.expanduser("~/.cache"), download=True, train=False)
    test_labels = torch.tensor(test_dataset.targets)
    test_sampler = BalancedBatchSampler(test_labels, BATCH_SIZE, 1)
    test_dataloader = torch.utils.data.DataLoader(test_dataset, drop_last=True)

    loss_img = torch.nn.CrossEntropyLoss()
    loss_txt = torch.nn.CrossEntropyLoss()

    # for p in model.transformer.parameters():
    #     p.requires_grad = False
    params = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.Adam(
        params, lr=1e-7, weight_decay=0.0001)

    num_batches_train = len(train_dataloader.dataset)/BATCH_SIZE
    num_batches_val = len(test_dataloader.dataset)/BATCH_SIZE

    for epoch in range(NUM_EPOCHS):
        print(f"Epoch: {epoch}")
        epoch_train_loss = 0
        model.train()
        for batch in tqdm(train_dataloader,total=num_batches_train):
            optimizer.zero_grad()

            images, class_ids = batch

            images = torch.stack([img for img in images], dim=0).to(
                device
            )
            # TODO: to use mean of multiple prompts need to pre-compute them.
            texts = [f"a photo of a {train_dataloader.dataset.classes[label_id]}" for label_id in class_ids]
            texts = clip.tokenize(texts).to(device)

            logits_per_image, logits_per_text = model(images, texts)

            ground_truth = torch.arange(logits_per_image.shape[0], dtype=torch.long, device=device)

            total_train_loss = (loss_img(logits_per_image, ground_truth) + loss_txt(logits_per_text, ground_truth)) / 2
            total_train_loss.backward()
            epoch_train_loss += total_train_loss

            torch.nn.utils.clip_grad_norm_(params, 1.0)

            if device == "cpu":
                optimizer.step()
            else:
                convert_models_to_fp32(model)
                optimizer.step()
                clip.model.convert_weights(model)

        epoch_train_loss /= num_batches_train
        writer.add_scalar("Loss/train", epoch_train_loss, epoch)

        if epoch % SAVE_INTERVAL == 0:
            torch.save(
                {
                    'epoch': epoch,
                    'model_state_dict': model.state_dict(),
                    'optimizer_state_dict': optimizer.state_dict(),
                    'loss': epoch_train_loss,
                }, weights_path / f"model_{epoch}.pt")  #just change to your preferred folder/filename
            print(f"Saved weights under model_checkpoint/model_{epoch}.pt.")

        # Compute test accuracy
        model.eval()
        values_list, indices_list = [], []
        top5_results = []
        top1_results = []
        acc_top1_list = []
        acc_top5_list = []

        num_batches_test = len(test_dataloader.dataset)/BATCH_SIZE
        epoch_test_loss = 0
        for i, batch in enumerate(tqdm(test_dataloader, total=num_batches_test)):
            images, class_ids = batch
            class_ids = class_ids.to(device)

            images = images.to(device)
            texts = torch.cat([clip.tokenize(f"a photo of a {c}") for c in test_dataloader.dataset.classes]).to(device)

            with torch.no_grad():
                # TODO: remove duplicate computation of image and text features
                image_features = model.encode_image(images)
                text_features = model.encode_text(texts)

                logits_per_image, logits_per_text = model(images, texts)
                ground_truth = torch.arange(logits_per_image.shape[0], dtype=torch.long, device=device)
                total_loss = (loss_img(logits_per_image, ground_truth) + loss_txt(logits_per_text, ground_truth)) / 2
                epoch_test_loss += total_loss

            image_features /= image_features.norm(dim=-1, keepdim=True)
            text_features /= text_features.norm(dim=-1, keepdim=True)
            similarity = (100.0 * image_features @ text_features.T).softmax(dim=-1)

            acc_top1 = torchmetrics.functional.accuracy(similarity, class_ids)
            acc_top5 = torchmetrics.functional.accuracy(similarity, class_ids, top_k=5)
            acc_top1_list.append(acc_top1)
            acc_top5_list.append(acc_top5)
        writer.add_scalar("Loss/test", epoch_test_loss / num_batches_test, epoch)

        print(f"Epoch {epoch} train loss: {epoch_train_loss / num_batches_train}")
        print(f"Epoch {epoch} test loss: {epoch_test_loss / num_batches_test}")

        # compute mean top5 accuracy and top1 accuracy
        mean_top5_accuracy = torch.stack(acc_top5_list).mean().cpu().numpy()
        print(f"Mean Top 5 Accuracy: {mean_top5_accuracy*100}%.")
        writer.add_scalar("Test Accuracy/Top5", mean_top5_accuracy , epoch)
        mean_top1_accuracy = torch.stack(acc_top1_list).mean().cpu().numpy()
        print(f"Mean Top 1 Accuracy: {mean_top1_accuracy*100}%.")
        writer.add_scalar("Test Accuracy/Top1", mean_top1_accuracy, epoch)

    writer.flush()
    writer.close()

if __name__ == '__main__':
    finetune()

Epoch: 0


  0%|          | 0/6250.0 [00:00<?, ?it/s]


TypeError: default_collate: batch must contain tensors, numpy arrays, numbers, dicts or lists; found <class 'PIL.Image.Image'>

In [ ]:
import os
import clip
import torch
from tqdm import tqdm
from torch.utils.data import DataLoader
from torchvision.datasets import FashionMNIST

# Define constants
LR = 1e-6 # Learning Rate
BATCH_SIZE = 32 # Number of images processed in one iteration
SHOTS = 5 # 5-way 5-shot
EPOCHS = 20 # Number of epochs

# Define controls
best_acc = 0.0 # Used to save best checkpoints

# Load the models
device = "cuda" if torch.cuda.is_available() else "cpu"
student, preprocess = clip.load('ViT-B/32', device, jit=False)
teacher, _ = clip.load('ViT-B/32', device)
for param in teacher.parameters():
    param.requires_grad = False

# Download the dataset
train_dataset = FashionMNIST(root=os.path.expanduser("~/.cache"), download=True, train=True, transform=preprocess)
test_dataset =  FashionMNIST(root=os.path.expanduser("~/.cache"), download=True, train=False, transform=preprocess)

# Prepare the data
train_dataloader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,       # Standard way to randomize
    drop_last=True      # Helpful for CLIP to keep matrix dimensions consistent
)
num_batches_train = len(train_dataloader) / BATCH_SIZE

all_class_texts = torch.cat([clip.tokenize(f"a photo of a {c}") for c in train_dataset.classes]).to(device)
NUM_CLASSES = len(train_dataset.classes) # This will be 10 for FashionMNIST

test_dataloader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,       # Standard way to randomize
    drop_last=True      # Helpful for CLIP to keep matrix dimensions consistent
)
num_batches_test = len(test_dataloader) / BATCH_SIZE

# Prepare criterion and optimizer
criterion = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(student.parameters(), lr=LR, weight_decay=0.1)

# Prepare the training
for epoch in range(EPOCHS):
    print(f"Epoch: {epoch}")
    epoch_train_loss = 0
    student.train() # Set the model to training mode (enables dropout/batchnorm updates)

    print("Running Training...")
    for batch in tqdm(train_dataloader, total=num_batches_train):
        optimizer.zero_grad() # Clear previous gradients before starting a new optimization step

        # Unpack images and class indices
        # images shape: [Batch, 3, 224, 224]
        # labels shape: [Batch]
        images, class_ids = batch
        images = images.to(device)

        # Map class IDs to text descriptions
        # We clean class names (replace underscores with spaces) for better CLIP performance
        texts = [f"a photo of a {train_dataset.classes[i].replace('_', ' ')}" for i in class_ids]
        texts = clip.tokenize(texts).to(device)

        # Forward pass
        # CLIP computes similarity between all images and all texts in the batch
        logits_per_image, logits_per_text = student(images, texts)

        # Define Ground Truth
        # Creates a diagonal target [0, 1, ..., N-1] where image i matches text i
        ground_truth = torch.arange(len(images), dtype=torch.long, device=device)

        # Compute Symmetric Loss
        loss_i = criterion(logits_per_image, ground_truth)
        loss_t = criterion(logits_per_text, ground_truth)
        total_loss = (loss_i + loss_t) / 2

        # Optimization
        total_loss.backward() # Perform backpropagation to calculate gradients
        optimizer.step() # Update model weights based on calculated gradients

        epoch_train_loss += total_loss.item()

    print(f"Epoch: {epoch} | Training Loss: {epoch_train_loss:.4f}")

    student.eval() # Switch to eval mode (disables dropout)
    acc_top1_list = []
    acc_top5_list = []

    print("Running Evaluation...")
    for batch in tqdm(test_dataloader, total=num_batches_test):
        images, class_ids = batch
        images = images.to(device)
        class_ids = class_ids.to(device)

        with torch.no_grad():
            # Encode images and all possible classes
            image_features = student.encode_image(images)
            text_features = student.encode_text(all_class_texts)

            # Normalize features (CLIP works best with unit vectors)
            image_features /= image_features.norm(dim=-1, keepdim=True)
            text_features /= text_features.norm(dim=-1, keepdim=True)

            # Calculate cosine similarity
            # [Batch Size, 512] @ [512, 100] -> [Batch Size, 100]
            logits_per_image = (100.0 * image_features @ text_features.T).softmax(dim=-1)

            # Calculate Accuracy using torchmetrics
            acc_top1 = torchmetrics.functional.accuracy(logits_per_image, class_ids, task="multiclass", num_classes=NUM_CLASSES, top_k=1)
            acc_top5 = torchmetrics.functional.accuracy(logits_per_image, class_ids, task="multiclass", num_classes=NUM_CLASSES, top_k=5)

            acc_top1_list.append(acc_top1)
            acc_top5_list.append(acc_top5)

    # Final Mean Accuracy
    epoch_acc_top1 = torch.stack(acc_top1_list).mean()
    epoch_acc_top5 = torch.stack(acc_top5_list).mean()

    print(f"Test Top-1 Acc: {epoch_acc_top1:.2%}")
    print(f"Test Top-5 Acc: {epoch_acc_top5:.2%}")

    # Save best model
    if epoch_acc_top1 > best_acc:
        best_acc = epoch_acc_top1
        torch.save(student.state_dict(), "best_model.pt")
        print("New best model saved!")

Epoch: 0
Running Training...


 99%|█████████▉| 58/58.59375 [00:12<00:00,  4.89it/s]/usr/local/lib/python3.12/dist-packages/tqdm/std.py:636: TqdmWarning: clamping frac to range [0, 1]
  full_bar = Bar(frac,
622it [02:10,  4.79it/s]